### Manual bundle: model -> checkpoint -> batch -> forward -> report

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Build the BEVFusion detection model from the resolved config.

Proves the config is self-consistent (camera 80ch + LiDAR 256ch -> ConvFuser 256,
decoder in 256, head in 512) and records parameter counts per submodule.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

rc, text = sc.run_script("step_model.py", log_name="step_model.log", timeout=1800)
report = sc.load_json("model.json")
print()
print("rc = %d" % rc)
print("params: %s" % json.dumps(report.get("model_params")))
print("init_cfg disabled (Swin pretrained download): %s" % report.get("init_cfg_disabled"))
print("init_cfg error: %s" % report.get("init_cfg_error"))
print("groups:")
for name in sorted(report.get("groups", {})):
    entry = report["groups"][name]
    print("  %-40s %-24s %12d" % (name, entry["type"], entry["params"]))
print("vtransform: %s" % json.dumps(report.get("vtransform"), sort_keys=True))
print("MODEL_CELL_OK" if "MODEL_OK" in text else "MODEL_CELL_FAILED (see logs/step_model.log)")


In [ ]:
"""Load the official fused checkpoint and list every key mismatch.

`step_checkpoint.py` builds the model, loads the checkpoint with
`load_state_dict(strict=False)` (a shape mismatch still raises, so it is caught
and reported), cross-checks with `mmcv.runner.load_checkpoint`, and verifies a
sample of weights really landed in the module.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

rc, text = sc.run_script("step_checkpoint.py", log_name="step_checkpoint.log", timeout=2400)
payload = sc.load_json("checkpoint.json")
ckpt = payload.get("checkpoint", {})
print()
print("rc = %d" % rc)
print("sha256        : %s" % ckpt.get("sha256"))
print("size_bytes    : %s" % ckpt.get("size_bytes"))
print("ckpt keys     : %s" % ckpt.get("ckpt_num_keys"))
print("model keys    : %s" % ckpt.get("model_num_keys"))
print("matched keys  : %s" % ckpt.get("matched_keys"))
print("shape mismatch: %s" % ckpt.get("shape_mismatch_count"))
print("missing keys  : %s (weight affecting: %s)"
      % (ckpt.get("missing_keys_total"), ckpt.get("missing_keys_weight_affecting_total")))
for key in ckpt.get("missing_keys", [])[:40]:
    print("   MISSING    " + key)
print("unexpected    : %s" % ckpt.get("unexpected_keys_total"))
for key in ckpt.get("unexpected_keys", [])[:40]:
    print("   UNEXPECTED " + key)
print("load_error    : %s" % ckpt.get("load_error"))
print("mmcv cross    : %s" % json.dumps(ckpt.get("cross_check_mmcv")))
print("weight sample : %s" % json.dumps(ckpt.get("weight_sample_check")))
print("CKPT_CELL_OK" if ckpt.get("ok") else "CKPT_CELL_FAILED (see logs/step_checkpoint.log)")


In [ ]:
"""Input batch: shapes, provenance, and why it is synthetic.

The Kaggle nuScenes mirror has no `nuscenes_infos_*.pkl` yet (open item of ticket
04), so no real sample can be pushed through the MIT pipeline. This cell records
that fact from the mounted dataset and prints the config-shaped synthetic batch
that the forward cell uses instead.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

system = sc.load_json("system.json")
dataset = system.get("dataset", {})
print("dataset root: %s" % dataset.get("root"))
print("infos: %s" % json.dumps(dataset.get("infos")))
print("real sample possible: %s" % dataset.get("real_sample_possible"))

rc, text = sc.run_script("smoke_batch.py", log_name="smoke_batch.log", timeout=900)
batch = sc.load_json("batch.json")
print()
print("rc = %d" % rc)
print("data_source: %s" % batch.get("data_source"))
print("why_synthetic: %s" % batch.get("why_synthetic"))
print("shapes:")
for key in sorted(batch.get("shapes", {})):
    print("  %-18s %s" % (key, json.dumps(batch["shapes"][key])))
sc.update(
    batch={
        "data_source": batch.get("data_source"),
        "shapes": batch.get("shapes"),
        "real_infos_available": bool(dataset.get("real_sample_possible")),
    }
)
print("BATCH_CELL_OK" if "BATCH_OK" in text else "BATCH_CELL_FAILED (see logs/smoke_batch.log)")


In [ ]:
"""Forward one sample: shapes, finiteness of every stage, depth-channel check.

Runs `step_forward.py` in the pinned venv. It hooks `dtransform` to capture the
exact depth tensor the transform receives, hooks the camera BEV, fused BEV,
decoder neck and head outputs, runs the model in eval mode under `no_grad`, and
records a pass/fail verdict per acceptance criterion.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

rc, text = sc.run_script("step_forward.py", log_name="step_forward.log", timeout=3600)
report = sc.load_json("forward.json")
print()
print("rc = %d" % rc)
print("batch shapes : %s" % json.dumps(report.get("batch", {}).get("shapes")))
print("forward      : %s" % json.dumps(report.get("forward")))
depth = report.get("depth_check", {})
print()
print("DEPTH CHECK")
for key in (
    "dtransform_first_layer",
    "dtransform_accepted_channels",
    "depth_tensor_shape",
    "depth_tensor_channels",
    "depth_tensor_nonzero_fraction",
    "depth_tensor_finite",
    "depth_tensor_min",
    "depth_tensor_max",
    "image_size",
    "depth_spatial_size",
    "depth_spatial_matches_image_size",
    "dtransform_output_size_computed",
    "feature_size",
    "dtransform_output_matches_feature_size",
    "channels_match",
    "V_D",
    "V_C",
    "class_source_lines",
    "forward_source_lines",
    "pass",
):
    print("  %-40s %s" % (key, json.dumps(depth.get(key))))
finite = report.get("finite_check", {})
print()
print("FINITE CHECK")
print("  stages : %s" % json.dumps(finite.get("stages")))
print("  head   : %s" % json.dumps(finite.get("head")))
print("  outputs: %s" % json.dumps(finite.get("outputs")))
print("  pass   : %s" % json.dumps(finite.get("pass")))
print()
print("OUTPUTS: %s" % json.dumps(report.get("outputs"), sort_keys=True))
print("VERDICT: %s" % json.dumps(report.get("pass"), sort_keys=True))
print("FORWARD_CELL_OK" if report.get("all_pass") else "FORWARD_CELL_FAILED (see logs/step_forward.log)")


In [ ]:
"""Aggregate every artifact into one pass/fail report.

Writes `artifacts/report.json` (machine-readable) and `smoke_test/REPORT.md`
(human-readable) into the notebook output, so the ticket can quote both.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

state = sc.state()
env = sc.load_json("env.json")
ops = sc.load_json("ops.json")
pin = sc.load_json("pin.json")
toolchain = sc.load_json("cuda_toolchain.json")
ckpt_file = sc.load_json("checkpoint_file.json")
cfg_checks = sc.load_json("cfg_checks.json")
model = sc.load_json("model.json")
ckpt = sc.load_json("checkpoint.json").get("checkpoint", {})
forward = sc.load_json("forward.json")

criteria = {
    "1_checkpoint_loads": bool(ckpt.get("ok")),
    "2_missing_unexpected_keys_listed": bool("missing_keys_total" in ckpt),
    "3_forward_no_shape_error": bool(forward.get("forward", {}).get("full_forward_ok")),
    "4_outputs_finite": bool(forward.get("finite_check", {}).get("pass")),
    "5_depth_channels_match": bool(forward.get("depth_check", {}).get("pass")),
    "6_report_saved": True,
}
supporting = {
    "env_pinned": bool(env.get("pinned_ok")),
    "ops_built_and_gpu_smoke": bool(ops.get("ok")),
    "pin_da93b27": bool(pin.get("ok")),
    "cuda_11x_toolkit": (toolchain.get("chosen") or {}).get("release"),
    "cuda_guard_bypassed": bool(toolchain.get("bypass_cuda_guard")),
    "checkpoint_obtained": bool(ckpt_file.get("ok")),
    "checkpoint_served_by": ckpt_file.get("served_by"),
    "config_invariants": bool(cfg_checks.get("ok")),
    "model_built": bool(model.get("model_params")),
}
report = {
    "ticket": "02-smoke-test-checkpoint",
    "pin": {"commit": pin.get("head"), "short": pin.get("head_short")},
    "env": env,
    "cuda_toolchain": toolchain.get("chosen"),
    "ops": {
        "ok": ops.get("ok"),
        "gpu_smoke": ops.get("gpu_smoke"),
    },
    "checkpoint": {
        "path": ckpt.get("path"),
        "sha256": ckpt.get("sha256"),
        "size_bytes": ckpt.get("size_bytes"),
        "served_by": ckpt_file.get("served_by"),
        "primary_url_probe": (ckpt_file.get("probes") or [{}])[0],
        "top_level_keys": ckpt.get("top_level_keys"),
        "ckpt_num_keys": ckpt.get("ckpt_num_keys"),
        "model_num_keys": ckpt.get("model_num_keys"),
        "matched_keys": ckpt.get("matched_keys"),
        "shape_mismatch_count": ckpt.get("shape_mismatch_count"),
        "shape_mismatch": ckpt.get("shape_mismatch"),
        "missing_keys_total": ckpt.get("missing_keys_total"),
        "missing_keys_weight_affecting_total": ckpt.get("missing_keys_weight_affecting_total"),
        "missing_keys": ckpt.get("missing_keys"),
        "unexpected_keys_total": ckpt.get("unexpected_keys_total"),
        "unexpected_keys": ckpt.get("unexpected_keys"),
        "load_error": ckpt.get("load_error"),
        "cross_check_mmcv": ckpt.get("cross_check_mmcv"),
        "weight_sample_check": ckpt.get("weight_sample_check"),
    },
    "batch": forward.get("batch"),
    "forward": forward.get("forward"),
    "depth_check": forward.get("depth_check"),
    "finite_check": {
        "stages": forward.get("finite_check", {}).get("stages"),
        "head": forward.get("finite_check", {}).get("head"),
        "outputs": forward.get("finite_check", {}).get("outputs"),
        "pass": forward.get("finite_check", {}).get("pass"),
    },
    "outputs": forward.get("outputs"),
    "criteria": criteria,
    "supporting": supporting,
    "all_pass": all(criteria.values()) and all(
        value for key, value in supporting.items() if key in ("env_pinned", "ops_built_and_gpu_smoke", "pin_da93b27", "checkpoint_obtained", "config_invariants")
    ),
}
sc.save_json("report.json", report)

lines = []
lines.append("# Smoke test report — pinned MIT BEVFusion (%s)" % (pin.get("head_short") or "?"))
lines.append("")
lines.append("- env: python %s, torch %s (cuda %s), mmcv-full %s, mmdet %s, numpy %s"
             % (env.get("python_version"), env.get("torch"), env.get("torch_cuda_build"),
                env.get("mmcv"), env.get("mmdet"), env.get("numpy")))
lines.append("- gpu: %s" % json.dumps(env.get("device_0")))
lines.append("- cuda toolkit for the build: %s" % json.dumps(toolchain.get("chosen")))
lines.append("- checkpoint: %s (%s bytes) sha256 %s served by %s"
             % (ckpt.get("path"), ckpt.get("size_bytes"), ckpt.get("sha256"), ckpt_file.get("served_by")))
lines.append("- config chain: %s" % ", ".join(os.path.basename(item["path"]) for item in sc.load_json("config_snapshot.json").get("chain", [])))
lines.append("")
lines.append("## Acceptance criteria")
for key in sorted(criteria):
    lines.append("- [%s] %s" % ("x" if criteria[key] else " ", key))
lines.append("")
lines.append("## Checkpoint keys")
lines.append("- checkpoint keys: %s, model keys: %s, matched: %s"
             % (ckpt.get("ckpt_num_keys"), ckpt.get("model_num_keys"), ckpt.get("matched_keys")))
lines.append("- shape mismatches: %s" % ckpt.get("shape_mismatch_count"))
lines.append("- missing: %s (weight affecting %s)" % (ckpt.get("missing_keys_total"), ckpt.get("missing_keys_weight_affecting_total")))
lines.append("- unexpected: %s" % ckpt.get("unexpected_keys_total"))
lines.append("")
lines.append("## Forward")
lines.append("- data source: %s (%s)" % (forward.get("batch", {}).get("data_source"), forward.get("batch", {}).get("why_synthetic")))
lines.append("- full forward ok: %s" % forward.get("forward", {}).get("full_forward_ok"))
lines.append("- seconds: %s, peak GPU memory: %s GB" % (forward.get("forward", {}).get("seconds"), forward.get("forward", {}).get("peak_gpu_mem_gb")))
lines.append("- depth tensor: %s" % json.dumps(forward.get("depth_check", {}).get("depth_tensor_shape")))
lines.append("- dtransform accepts %s channels, depth provides %s"
             % (forward.get("depth_check", {}).get("dtransform_accepted_channels"),
                forward.get("depth_check", {}).get("depth_tensor_channels")))
lines.append("- finite: stages=%s outputs=%s"
             % (json.dumps(forward.get("finite_check", {}).get("stages")),
                json.dumps(forward.get("finite_check", {}).get("outputs"))))
lines.append("")
with open(os.path.join(WORK, "REPORT.md"), "w") as handle:
    handle.write("\n".join(lines))

print("\n".join(lines))
print()
print("SUMMARY support: %s" % json.dumps(supporting, sort_keys=True))
print("SUMMARY criteria: %s" % json.dumps(criteria, sort_keys=True))
print("ALL_PASS" if report["all_pass"] else "NOT_ALL_PASS")
